# Planet Habitable Zone Prediction

The habitable zone is the range of orbital distances from a star where liquid water could exist on a planet's surface. Whether a planet lies in this zone depends on things like equilibrium temperature, orbital period, and the properties of the host star.

This is a binary classification problem. We are predicting whether an exoplanet is in the habitable zone (1.0) or not (0.0). The data is heavily imbalanced since only about 7% of planets are habitable.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
X_cleaned = pd.read_csv('./data/X_Habitable.csv')
y_cleaned = pd.read_csv('./data/Y_Habitable.csv')

X_cleaned = X_cleaned.drop(columns=['Unnamed: 0'])
y_cleaned = y_cleaned.drop(columns=['Unnamed: 0'])

X_cleaned

## Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(16, 12))
y_cleaned.value_counts().sort_index().plot(kind='bar', title='Habitable Zone Classification')
plt.xlabel('Habitable Zone Flag (0 = Not Habitable, 1 = Habitable)')
plt.ylabel('Frequency')
plt.show()

There is a large class imbalance here. Only about 7% of planets are in the habitable zone. Using PCA and KMeans to check if there is any natural clustering in the data:

## Unsupervised Clustering

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

In [ ]:
pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X_cleaned)

X_train, X_test, y_train, y_test = train_test_split(X_reduced, y_cleaned, test_size=0.2, random_state=42)

kmeans = KMeans(n_clusters=2, random_state=0, n_init="auto")
kmeans.fit(X_train)
y_kmeans = kmeans.predict(X_test)

plt.scatter(X_test[:, 0], X_test[:, 1], c=y_kmeans, s=50, cmap='viridis')

centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c='black', s=200, alpha=0.5, marker='X', label='Centroids')
plt.legend()
plt.show()

Comparing the KMeans clusters against the actual habitable zone labels:

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import silhouette_score

In [ ]:
cm = confusion_matrix(y_test, y_kmeans)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
silhouette_score(X_test, y_kmeans)

Trying KMeans again without PCA to see if using all features helps:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_cleaned, y_cleaned, test_size=0.2, random_state=42)

kmeans2 = KMeans(n_clusters=2, random_state=0, n_init="auto")
kmeans2.fit(X_train)
y_kmeans2 = kmeans2.predict(X_test)

In [ ]:
cm = confusion_matrix(y_test, y_kmeans2)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## Classification

### K-Nearest Neighbors

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_cleaned, y_cleaned, test_size=0.2, random_state=42)

In [ ]:
neigh = KNeighborsClassifier(n_neighbors=3)
neigh.fit(X_train, y_train.values.ravel())
y_pred = neigh.predict(X_test)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')

### K-Nearest Neighbors with SMOTE

Using SMOTE to oversample the minority class in the training data to address the class imbalance.

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_cleaned, y_cleaned, test_size=0.2, random_state=42)

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)

neigh = KNeighborsClassifier(n_neighbors=3)
neigh.fit(X_res, y_res.values.ravel())
y_pred = neigh.predict(X_test)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_cleaned, y_cleaned, test_size=0.2, random_state=42)

In [ ]:
clf = RandomForestClassifier(max_depth=10, random_state=0)
clf.fit(X_train, y_train.values.ravel())
y_pred = clf.predict(X_test)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()